# 03 - Assign Cluster Labels to ALL Trajectories  (Stage 3)

**Purpose.** Load the chosen k-means model and assign a `cluster_label` to every
trajectory by nearest-centroid lookup (`predict`, no iteration).

- `complete` and `partial` trajectories are labelled (for `partial`, the last
  known position was already used as the day-100 proxy in Stage 1).
- `early_loss` trajectories have no usable features -> `cluster_label = -1`.

If `config.GROUP_MAP` is set, a merged `cluster_group` column is also added
(raw clusters merged into named pathway groups); otherwise `cluster_group`
mirrors `cluster_label`.

**Input.** `data/features.parquet`, `data/kmeans_models/kmeans_k{BEST_K}.pkl`.
**Output.** `data/labeled_trajectories.parquet`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)

Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)/home/b/b383184/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


project root: /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis


## 3.1  Choose k and load the model

In [2]:
BEST_K = 20          # <-- set after inspecting notebook 02 (papermill: -p BEST_K <k>)

In [3]:
# Parameters
BEST_K = 20


In [4]:
import pickle
with open(C.MODELS_DIR / f"kmeans_k{BEST_K}.pkl", "rb") as f:
    km = pickle.load(f)
print("loaded model with", km.n_clusters, "clusters")

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


loaded model with 20 clusters


/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator MiniBatchKMeans from version 1.7.0 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 3.2  Predict labels for every labellable trajectory

In [5]:
features = pd.read_parquet(C.FEATURES_FILE)
labelable = features.status != "early_loss"
X_all = P.build_feature_matrix(features[labelable])
labels = np.full(len(features), -1, dtype=np.int32)
labels[labelable.to_numpy()] = km.predict(X_all).astype(np.int32)
features["cluster_label"] = labels
print(features.cluster_label.value_counts().sort_index().to_string())

/sw/spack-levante/mambaforge-23.1.0-1-Linux-x86_64-3boc6i/lib/python3.10/site-packages/threadpoolctl.py:762: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


cluster_label
-1      200000
 0      339019
 1     4870105
 2      411954
 3     1092355
 4      634510
 5      795673
 6      356298
 7       77649
 8      364375
 9      141877
 10     645618
 11     222408
 12    1546895
 13    2252002
 14     393855
 15     512087
 16     189985
 17     499887
 18     183299
 19     550149


## 3.2b  Merge into pathway groups (optional)

If `config.GROUP_MAP` is non-empty, merge the raw clusters into groups; the
result is stored in `cluster_group`. With an empty map, `cluster_group` simply
equals `cluster_label`.

In [6]:
if C.GROUP_MAP:
    groups, raw2grp = P.apply_group_map(features.cluster_label.to_numpy(), C.GROUP_MAP, BEST_K)
    features["cluster_group"] = groups.astype(np.int32)
    print("raw cluster -> group:", raw2grp)
    print(features.loc[features.cluster_group >= 0, "cluster_group"]
          .value_counts().sort_index().to_string())
else:
    features["cluster_group"] = features["cluster_label"]
    print("GROUP_MAP empty -> cluster_group mirrors cluster_label")

GROUP_MAP empty -> cluster_group mirrors cluster_label


## 3.3  Save labelled trajectories

In [7]:
features.to_parquet(C.LABELED_FILE, index=False)
print("saved", features.shape, "->", C.LABELED_FILE)

saved (16280000, 12) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/data/labeled_trajectories.parquet


## 3.4  Summary

In [8]:
lbl = features[features.cluster_label >= 0]
print(f"Labelled {len(lbl):,} trajectories into {BEST_K} clusters "
      f"({(features.cluster_label==-1).sum():,} early_loss left unlabelled).")
print(f"Saved to {C.LABELED_FILE}")

Labelled 16,080,000 trajectories into 20 clusters (200,000 early_loss left unlabelled).
Saved to /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/data/labeled_trajectories.parquet
